# Dumbarton FC — Sons Sponsors · Colab runner

Build the ranked sponsor list from **real local-business data**, in Colab, with no local setup.
Google Places and Companies House are official APIs, so they work from Colab's servers.

**Run the cells top to bottom.** Without keys, everything runs on built-in test data so you can
try the whole flow first. The live run only starts when you switch it on in step 6.

| Step | What it does |
|---|---|
| 1 | Get the code (and apply the update zip until it's pushed to GitHub) |
| 2–3 | Install, and load your API keys from Colab Secrets |
| 4 | Optional: keep the database in Google Drive |
| 5 | Your sponsor list (`connections.csv`), optionally upload the one downloaded from the page |
| 6 | Check what a live run will do, then switch it on |
| 7 | Build the list |
| 8–10 | What it found, inspect a business, AI leads to confirm |
| 11–13 | Record outreach, build the club website, download everything |

## 1 · Get the code

In [ ]:
import os
REPO_URL = "https://github.com/th1555/TH_DFC_Sponsors.git"
os.chdir("/content" if os.path.isdir("/content") else os.getcwd())
START_DIR = os.getcwd()

if os.path.exists("sponsor_tool"):
    !git -C sponsor_tool pull -q
else:
    !git clone -q "$REPO_URL" sponsor_tool
os.chdir("sponsor_tool")
print("working dir:", os.getcwd())

### 1b · Apply the update (only until it's pushed to GitHub)
If the repo doesn't have the new files yet, this asks you to upload **`TH_DFC_Sponsors_update.zip`**
(or picks it up if it's already in `/content`) and copies the new code over the clone.
Once you've pushed the update to GitHub, this cell just says "already up to date".

In [ ]:
import glob, shutil, zipfile

def apply_update(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall("/tmp/update")
    root = next(d for d, _, files in os.walk("/tmp/update") if "data_checks.py" in files)
    copied = []
    for f in os.listdir(root):
        src = os.path.join(root, f)
        if os.path.isfile(src):
            shutil.copy(src, f); copied.append(f)
    print(f"applied update from {os.path.basename(zip_path)}: {len(copied)} files")

if os.path.exists("data_checks.py"):
    print("already up to date (the repo has the new code)")
else:
    zips = sorted(glob.glob(os.path.join(START_DIR, "TH_DFC_Sponsors_update*.zip")) + glob.glob("TH_DFC_Sponsors_update*.zip"))
    if not zips:
        try:
            from google.colab import files
            print("Upload TH_DFC_Sponsors_update.zip:")
            zips = [k for k in files.upload() if k.endswith(".zip")]
        except ImportError:
            pass
    if zips:
        apply_update(zips[-1])
    else:
        raise SystemExit("No update zip found. Upload TH_DFC_Sponsors_update.zip and re-run this cell.")

## 2 · Install

In [ ]:
!pip install -q pandas numpy pyyaml requests matplotlib lxml
print("dependencies ready")

## 3 · API keys (Colab Secrets)
Click the **🔑 key icon** in the left sidebar, add these as secrets, and turn on *Notebook access*.
Never paste a key into a cell.

| Secret | Cost | Where |
|---|---|---|
| `COMPANIES_HOUSE_API_KEY` | free | developer.company-information.service.gov.uk |
| `GOOGLE_PLACES_API_KEY` | needs Cloud billing; check current Places pricing | console.cloud.google.com → Places API (New) |
| `ANTHROPIC_API_KEY` | paid, small for this volume | console.anthropic.com |
| `SPONSOR_LLM_MODEL` | optional | override the default model (`claude-haiku-5-5`) |

In [ ]:
try:
    from google.colab import userdata
    for k in ["GOOGLE_PLACES_API_KEY", "COMPANIES_HOUSE_API_KEY", "ANTHROPIC_API_KEY", "SPONSOR_LLM_MODEL"]:
        try:
            v = userdata.get(k)
            if v:
                os.environ[k] = v
        except Exception:
            pass
except ImportError:
    pass  # not in Colab

has_places_ch = bool(os.environ.get("GOOGLE_PLACES_API_KEY") and os.environ.get("COMPANIES_HOUSE_API_KEY"))
has_llm = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("Places + Companies House keys:", "found" if has_places_ch else "missing (test data)")
print("Anthropic key:                ", "found" if has_llm else "missing (keyword stand-in)")
print("AI model:                     ", os.environ.get("SPONSOR_LLM_MODEL", "claude-haiku-5-5 (default)"))

## 4 · (Optional) Keep the database in Google Drive
Colab's disk is wiped when the runtime recycles. Turn this on to keep `sponsor.db` (and any
outreach notes) between sessions.

In [ ]:
USE_DRIVE = False   # set True to keep the database in your Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/dumbarton_sponsor", exist_ok=True)
    os.environ["DUMBARTON_SPONSOR_DB"] = "/content/drive/MyDrive/dumbarton_sponsor/sponsor.db"
    print("database ->", os.environ["DUMBARTON_SPONSOR_DB"])
else:
    print("using a temporary sponsor.db (fine for one run)")

## 5 · Your sponsor list (`connections.csv`)
Current sponsors, past sponsors, warm paths and "not a fit" businesses. The repo copy is seeded
from the club's 2026/27 commercial page. If someone has tagged businesses on the website, click
**Sponsor list** there, then set `UPLOAD_CONNECTIONS = True` and run this cell to use that file.

In [ ]:
import pandas as pd
UPLOAD_CONNECTIONS = False

if UPLOAD_CONNECTIONS:
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    new = pd.read_csv(name)
    if list(new.columns[:3]) != ["match", "kind", "note"]:
        raise SystemExit(f"{name} doesn't look like a sponsor list (expected columns: match, kind, note)")
    if name != "connections.csv":
        shutil.move(name, "connections.csv")
    print(f"using the uploaded sponsor list: {len(new)} rows")

conns = pd.read_csv("connections.csv")
print(conns.kind.value_counts().to_string(), "\n")
conns

## 6 · Check before going live
Shows what a live run will do. **Nothing live happens until you set `GO_LIVE = True`.**
Without that (or without keys) the build uses test data.

In [ ]:
import yaml
cfg = yaml.safe_load(open("sponsor_config.yaml"))
pl = cfg["ingest"]["places"]
n_q, n_towns = len(pl["seed_queries"]), len(pl.get("towns") or [None])
n_search = n_q * n_towns
est_low, est_high = n_search * 3, n_search * 6      # unique businesses after de-duplication (rough)

print(f"Google Places searches: {n_q} queries x {n_towns} towns = {n_search}")
print(f"  each returns up to {pl['max_per_query']} places; expect very roughly {est_low}-{est_high} unique businesses")
print(f"  catchment: up to {pl['max_distance_km']} km from {cfg['ingest']['stadium']['name']}")
print(f"Companies House: ~3 calls per business, paced to stay under its rate limit")
print(f"  -> roughly {est_low*3*0.55/60:.0f}-{est_high*3*0.55/60:.0f} minutes for this step")
print(f"Accounts (employees, turnover): ~3 more calls per linked company, same pacing")
print(f"  -> roughly {est_low*0.5*3*0.55/60:.0f}-{est_high*0.5*3*0.55/60:.0f} extra minutes (about half the businesses have a company record)")
print(f"AI readings: one call per business with a website, 8 at a time")
print("\nCosts: Companies House is free. Check current Google Places pricing for "
      f"{n_search} Text Search requests (the rating fields may bill at a higher tier) "
      "and your Anthropic usage page afterwards.")

GO_LIVE = False      # <- set True when you're ready to spend on the live APIs

## 7 · Build the list

In [ ]:
import importlib, time
import sponsor_db as db;        importlib.reload(db)
import sponsor_pipeline as P;   importlib.reload(P)

ingest_mode = "api" if (GO_LIVE and has_places_ch) else "fixture"
soft_mode   = "llm" if (GO_LIVE and has_llm) else "stub"
if GO_LIVE and not has_places_ch:
    print("GO_LIVE is on but the Places / Companies House keys are missing: using test data")

t0 = time.time()
ACCOUNTS = True      # employees + revenue from filed accounts (free; adds a few minutes)
conn = P.run(mode=soft_mode, source="real", ingest_mode=ingest_mode, accounts=ACCOUNTS)
q = db.get_queue(conn)
print(f"\nbuilt {len(q)} businesses in {(time.time()-t0)/60:.1f} min  |  discovery={ingest_mode}  |  AI={soft_mode}")

### 7b · Employees and revenue only (optional)
Already have a `sponsor.db` from an earlier run (kept in Drive, step 4)? This adds the
employees and revenue figures to it **without** re-running the Places search or the AI
readings, so it costs nothing. Set `ACCOUNTS_ONLY = True` and run it instead of step 7.


In [ ]:
ACCOUNTS_ONLY = False

if ACCOUNTS_ONLY:
    import sponsor_db as db;        importlib.reload(db)
    import sponsor_pipeline as P;   importlib.reload(P)
    conn = db.connect(); db.init_db(conn)
    if not conn.execute("SELECT COUNT(*) FROM businesses").fetchone()[0]:
        print("No businesses in sponsor.db yet: run step 7 first.")
    else:
        mode = "api" if (GO_LIVE and has_places_ch) else "fixture"
        P.refresh_accounts(conn, mode)
        q = db.get_queue(conn)


## 8 · What the run found

In [ ]:
b = pd.read_sql("SELECT * FROM businesses", conn)
s = q.merge(b[["business_id", "connection_kind", "company_number", "directors_local",
               "google_reviews", "ch_flag", "size_band"]], on="business_id", how="left")
summary = {
    "businesses": len(s),
    "within 5 km": int((s.distance_km <= 5).sum()),
    "within 10 km": int((s.distance_km <= 10).sum()),
    "national chains / franchises": int((s.sector == "national_chain").sum()),
    "linked to a Companies House record": int(s.company_number.notna().sum()),
    "Companies House match rejected (wrong place)": int(s.ch_flag.fillna("").str.contains("outside Scotland").sum()),
    "with a director based locally": int((s.directors_local.fillna(0) > 0).sum()),
    "with Google reviews": int(s.google_reviews.notna().sum()),
    "employees reported (filed accounts)": int(pd.read_sql("SELECT COUNT(*) n FROM revenue_estimates WHERE employees_basis='reported'", conn).n[0]),
    "revenue reported / estimated": "{} / {}".format(*[int(pd.read_sql(f"SELECT COUNT(*) n FROM revenue_estimates WHERE revenue_basis='{k}'", conn).n[0]) for k in ("reported", "estimate")]),
    "AI leads to confirm (step 10)": int(s["flags"].fillna("").str.contains("possible connection").sum()),
}
for k, v in summary.items():
    print(f"  {k:<46} {v}")
print("\nrecorded connections:", s.connection_kind.value_counts().to_dict() or "none")

prospects = s[~s.connection_kind.isin(["current_sponsor", "not_a_fit"]) & (s.sector != "national_chain")]
print(f"\nTop 15 prospects (current sponsors, 'not a fit' and chains left out):")
(prospects[["name", "sector", "fit_score", "distance_km", "size_band", "tier"]].head(15)
   .style.bar(subset=["fit_score"], vmin=0, vmax=100, color="#c8102e")
   .format({"fit_score": "{:.0f}", "distance_km": "{:.1f}"}).hide(axis="index"))

## 9 · Inspect a business
Every score breaks down into its parts; each AI reading quotes the business's own website.

In [ ]:
def inspect(name):
    row = q[q.name == name].iloc[0]
    biz, sfeat, contrib = db.get_detail(conn, row.business_id)
    print(f"{name}  —  fit {row.fit_score:.0f}  |  {row.sector}  |  {row.distance_km:.1f} km")
    print(f"approach: {row.tier}")
    if biz.get("connection_kind"):
        print(f"connection: {biz['connection_kind']} — {biz['connection_note']}")
    print(f"checks:   {row.flags}")
    print("\npoints (out of each part's weight):")
    for k, v in sorted(contrib.items(), key=lambda x: -x[1]):
        print(f"   {k.replace('_',' '):<20} {v:5.1f}")
    print("\nwebsite readings:")
    for _, r in sfeat.iterrows():
        print(f"   {r.feature:<20} value={r.value:.2f} conf={r.confidence:.2f}  \"{r.justification}\"")

try:
    import ipywidgets as w
    w.interact(inspect, name=w.Dropdown(options=list(q.name), description="Business"))
except Exception:
    inspect(q.iloc[0]["name"])

## 10 · AI leads to confirm
Businesses whose website mentions sponsoring a team or the club. These are **not scored**:
check each one, then tag it on the website (or add it to `connections.csv`) so it counts.

In [ ]:
leads = s[s["flags"].fillna("").str.contains("possible connection")]
if leads.empty:
    print("No leads" + ("" if soft_mode == "llm" else " (the keyword stand-in rarely finds any; the real AI run does)"))
else:
    for _, r in leads.iterrows():
        lead = next(f for f in r["flags"].split("; ") if f.startswith("possible connection"))
        print(f"- {r['name']} ({r.distance_km:.1f} km): {lead}")

## 11 · Record outreach (optional)
Off by default. Set `RECORD = True` to add a status or note; it's kept separate from the scores,
so rebuilding never wipes it (turn on step 4 to keep it between sessions).

In [ ]:
RECORD = False
TARGET = "Business name here"      # exact name, as in the step 8 table
STATUS = "to-approach"             # to-approach, contacted, in-talks, won, dead
OWNER  = "Tom"
NOTE   = ""

if RECORD:
    match = q[q.name == TARGET]
    if match.empty:
        print(f"No business called {TARGET!r}. Check the spelling against step 8.")
    else:
        db.set_status(conn, match.iloc[0].business_id, STATUS, owner=OWNER, notes=NOTE)
        print(f"Recorded: {TARGET} -> {STATUS}")
else:
    print("Not recording anything (RECORD = False).")

## 12 · Build the club website
`site/index.html` is the page the club uses; `site/lab.html` adds the website readings.
The preview is the real page: open a business to tag it, or try the filters.
(Tags made inside this preview may not save; use the deployed page for that.)

In [ ]:
import html
from IPython.display import HTML, display
!python export_sponsor_site.py

PAGE = "index.html"        # or "lab.html"
page = open(os.path.join("site", PAGE), encoding="utf-8").read()
display(HTML(f'<iframe srcdoc="{html.escape(page)}" style="width:100%;height:820px;border:1px solid #ddd;border-radius:8px"></iframe>'))

## 13 · Download
One zip with the website (`site/`) and the sponsor list used for this run. Put the two pages
wherever the club views them. Before anyone records outreach notes, host them behind a login
(the README's Cloudflare Access setup), not in a public repo.

In [ ]:
os.makedirs("download", exist_ok=True)
shutil.copytree("site", "download/site", dirs_exist_ok=True)
shutil.copy("connections.csv", "download/connections.csv")
shutil.make_archive("/tmp/sons_sponsors_output", "zip", "download")
try:
    from google.colab import files
    files.download("/tmp/sons_sponsors_output.zip")
except ImportError:
    print("zipped to /tmp/sons_sponsors_output.zip")